#Basic Tasks

In [0]:
#1
data = [
    (101, "John", 25),
    (102, "Jane", 30),
    (103, "Bob", 35)
]

schema = ["id", "name", "age"]

df = spark.createDataFrame(data, schema)

In [0]:
#2
customers = spark.read.csv('/Volumes/cyntexa_dev/sales/raw/customers_20260901.csv' ,header = True, inferSchema = True)
customers.printSchema()

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
schema = StructType([
    StructField("employee_id", IntegerType(), False),
    StructField("first_name", StringType(), True),     
    StructField("last_name", StringType(), True)
])

data = [
    (101, "Alice", "Smith"),
    (102, "Bob", None)
]

emp = spark.createDataFrame(data, schema=schema)
emp.printSchema()

inferSchema=True — spark scans the data itself to guess column types automatically it is easy but slower

StructType — needs manually define exact column names + types upfront faster and more reliable no guessing best for production pipelines where schema stay consistent

In [0]:
#3
emp.show()

the output does't show before any action is perfomed as it follows the lazy evaluation which not allows the data to be displayed

#Intermediate Tasks

In [0]:
from pyspark.sql.functions import current_timestamp

df = spark.read.csv('/Volumes/cyntexa_dev/sales/raw/customers_20260901.csv')
df = df.withColumn('ingestion_date', current_timestamp())

In [0]:
from pyspark.sql.functions import explode, col
df_p = spark.read.option('multiline', 'true').json('/Volumes/cyntexa_dev/sales/raw/sample_products (1).json')
display(df)

In [0]:
df_exploded = df_p.select(explode("products").alias("p"))
display(df_exploded)

In [0]:
df_flat = df_exploded.select(
    col("p.id").alias("id"),
    col("p.name").alias("name"),
    col("p.category").alias("category"),
    col("p.price.amount").alias("price_amount"),              
    col("p.price.currency").alias("price_currency"),          
    col("p.specifications.color").alias("color"),             
    col("p.specifications.weight").alias("weight"),           
    col("p.specifications.batteryLife").alias("battery_life"),
    col("p.inStock").alias("in_stock"),
    col("p.stockCount").alias("stock_count")
)

display(df_flat)

In [0]:
#6
from pyspark.sql.functions import explode, col, avg, count

df_exploded = df.select(explode("products").alias("p"))

df_flat = df_exploded.select(
    col("p.id").alias("id"),
    col("p.category").alias("category"),
    col("p.price.amount").alias("price_amount"),
    col("p.inStock").alias("in_stock"),
    col("p.stockCount").alias("stock_count")
)


result = (
    df_flat
    .groupBy("category")                          
        .agg(
        avg("price_amount").alias("avg_price"),
        count("id").alias("product_count")
    )
    .orderBy(col("avg_price").desc())             
)

result.explain(True)

explain() shows the physical plan of a query wherever the word exchange appears that means a shuffle happened shuffle means data moved between different partitions

steps between exchange steps get pipelined together meaning they run one after another without moving data, so it stays fast

simple rule: filter select and explode usually get pipelined groupby orderby and join usually cause a shuffle

#Advanced Tasks

In [0]:
#7
import pandas as pd
with open('/Volumes/cyntexa_dev/sales/raw/sample_products (1).json') as f:
    data = json.load(f)

df = pd.DataFrame(data["products"]) 

df = df[df["inStock"] == True]

df["price_amount"] = df["price"].apply(lambda x: x["amount"])
df["price_currency"] = df["price"].apply(lambda x: x["currency"])

summary = df.groupby("category").agg(
    avg_price=("price_amount", "mean"),
    product_count=("price_amount", "count")
).reset_index()

summary = summary.sort_values("avg_price", ascending=False)

print(summary)

In [0]:
from pyspark.sql.functions import explode, col, avg, count

df = spark.read.option("multiline", "true").json('/Volumes/cyntexa_dev/sales/raw/sample_products (1).json')

df_exploded = df.select(explode("products").alias("p"))

df_flat = df_exploded.select(
    col("p.category").alias("category"),
    col("p.price.amount").alias("price_amount"),
    col("p.inStock").alias("in_stock")
)

summary = (
    df_flat
    .filter(col("in_stock") == True)
    .groupBy("category")
    .agg(
        avg("price_amount").alias("avg_price"),
        count("price_amount").alias("product_count")
    )
    .orderBy(col("avg_price").desc())
)

display(summary)

Pandas loads the dataFrame into the driver's memory and performs the operation on one machine and one machine reads and processes the file or it needs sufficient memory for both dataframes.

Spark distributes data across multiple workers and the aggregation using a shuffle it can read files in parallel and the join are across executors

In [0]:
#8
from pyspark.sql import functions as F

sales_df = spark.read.option("header", True).option("inferSchema", True).csv('/Volumes/dev/bronze/raw/sales/')
sales_df = (
    sales_df
    .withColumn("year", F.year("order_date"))
    .withColumn("month", F.month("order_date"))
)

the sales table is mostly queried by date range partition it by year and month using partitionBy("year", "month")

avoid high-cardinality columns such as customer_id because they can create too many small partitions and target reasonably sized files like 128 MB to 1 GB

Spark transformations are lazy so a filter does not immediately execute when an action such as show() or count() runs spark creates a physical plan or perform action and Spark can perform partition pruning and read only the relevant date partitions.

In [0]:
#9
from pyspark.sql import functions as F

df = spark.read.option("header", True).option("inferSchema", True).csv('/Volumes/dev/bronze/raw/sales/')
df = df.withColumn("sale_date", F.to_date("sale_date"))
df = df.withColumn("month", F.date_format("sale_date", "yyyy-MM"))
df = df.dropna()
df = df.dropDuplicates()

monthly_revenue = (
    df
    .groupBy("month", "product_id")
    .agg(
        F.sum("sale_amount").alias("total_revenue"),
        F.sum("quantity").alias("total_quantity"),
        F.countDistinct("sale_id").alias("total_orders")
    )
    .orderBy("month", "product_id")
)
monthly_revenue.show()

In [0]:
monthly_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("cyntexa_dev.sales.monthly_revenue")
     